In [1]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/ngocccchannn/cold-scenario/cold_items_test.json
/kaggle/input/datasets/ngocccchannn/cold-scenario/cold_items_val.json
/kaggle/input/datasets/ngocccchannn/embeddings-course-index/embeddings_course_index.json
/kaggle/input/datasets/ngocccchannn/llm-3d/llm_profile_no_prereq_followup_embeddings.npy
/kaggle/input/datasets/ngocccchannn/raw-embeddings/raw_course_text_embeddings.npy
/kaggle/input/datasets/ngocccchannn/raw-embeddings/raw_embeddings_course_index.json
/kaggle/input/datasets/ngocccchannn/lightgcn-backbone-cold/user_index_cold.json
/kaggle/input/datasets/ngocccchannn/lightgcn-backbone-cold/g_u.npy
/kaggle/input/datasets/ngocccchannn/lightgcn-backbone-cold/g_i.npy
/kaggle/input/datasets/ngocccchannn/lightgcn-backbone-cold/item_index.json
/kaggle/input/datasets/ngocccchannn/evaluator/evaluator.py
/kaggle/input/datasets/ngocccchannn/llm-embeddings/llm_profile_embeddings.npy
/kaggle/input/datasets/ngocccchannn/llm-embeddings/full_3769_profiles.json
/kaggle/input/

In [2]:
import os
import copy
import hashlib
import json
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import sys

# Thiết lập seed cố định đảm bảo tính tái lập
def set_seed(s=42):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)
set_seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Sử dụng thiết bị: {DEVICE}")
print("[Info] PyTorch:", torch.__version__)
print("[Info] CUDA khả dụng:", torch.cuda.is_available())

# Bổ sung đường dẫn module evaluator
sys.path.append('/kaggle/input/datasets/ngocccchannn/evaluator')
from evaluator import evaluate, topk_random_ties

Sử dụng thiết bị: cuda
[Info] PyTorch: 2.11.0+cu128
[Info] CUDA khả dụng: True


In [3]:
# Nạp tệp 
RAW_NPY = '/kaggle/input/datasets/ngocccchannn/raw-embeddings/raw_course_text_embeddings.npy'
raw_course_text_embeddings = np.load(RAW_NPY)
print(f'Kích thước của embedding văn bản gốc: {raw_course_text_embeddings.shape}')

with open('/kaggle/input/datasets/ngocccchannn/lightgcn-backbone-cold/item_index.json', 'r') as f:
    course_ids = json.load(f)
assert len(course_ids) == 3769 and course_ids == sorted(course_ids), "Lỗi item_index.json"
assert course_ids == json.load(open("/kaggle/input/datasets/ngocccchannn/raw-embeddings/raw_embeddings_course_index.json"))["course_id"]

with open('/kaggle/input/datasets/ngocccchannn/lightgcn-backbone-cold/user_index_cold.json', 'r') as f:
    user_cold_ids = json.load(f)

print(f'Tổng số khóa học: {len(course_ids)}, Tổng số học viên (khóa lạnh): {len(user_cold_ids)}')

G_U_PATH = '/kaggle/input/datasets/ngocccchannn/lightgcn-backbone-cold/g_u.npy'
G_I_PATH = '/kaggle/input/datasets/ngocccchannn/lightgcn-backbone-cold/g_i.npy'

g_i_cold = np.load(G_I_PATH)
g_u_cold = np.load(G_U_PATH)
print(f"Bảng kích thước: g_u_cold = {g_u_cold.shape}, g_i_cold = {g_i_cold.shape}")
assert g_u_cold.shape == (len(user_cold_ids), 64) and g_i_cold.shape == (len(course_ids), 64), "Lỗi kích thước embedding nền chung"

# In MD5 của backbone và embedding văn bản gốc
def file_md5(path):
    return hashlib.md5(open(path, 'rb').read()).hexdigest()

print(f"MD5 g_u.npy: {file_md5(G_U_PATH)}")
print(f"MD5 g_i.npy: {file_md5(G_I_PATH)}")
print(f"MD5 raw_course_text_embeddings.npy: {file_md5(RAW_NPY)}")

Kích thước của embedding văn bản gốc: (3769, 384)
Tổng số khóa học: 3769, Tổng số học viên (khóa lạnh): 98220
Bảng kích thước: g_u_cold = (98220, 64), g_i_cold = (3769, 64)
MD5 g_u.npy: 7a170558642de73d301970e8ce7f0a86
MD5 g_i.npy: d17ad9dfc4f4d6cb6de915c8649070d7
MD5 raw_course_text_embeddings.npy: f80445649ef8a8217d048bd3e7ac0ad5


In [4]:
# 2. Nạp embedding văn bản gốc (để kết hợp tạo fused_3d) và embedding LLM 3 chiều
LLM_3D_NPY = "/kaggle/input/datasets/ngocccchannn/llm-3d/llm_profile_no_prereq_followup_embeddings.npy"
print(f"MD5 llm_profile_no_prereq_followup_embeddings.npy: {file_md5(LLM_3D_NPY)}")

S_raw = np.load(RAW_NPY).astype(np.float32)
s_llm_3d = np.load(LLM_3D_NPY).astype(np.float32)

# Hàm chuẩn hóa L2
nz = lambda X: X / np.linalg.norm(X, axis=1, keepdims=True).clip(1e-9)

S_raw_norm = nz(S_raw) # [3769, 384]
S_llm_3d_norm = nz(s_llm_3d) # [3769, 384]
S_fused_3d = np.concatenate([S_raw_norm, S_llm_3d_norm], axis=1) # [3769, 768]

assert S_llm_3d_norm.shape == (3769, 384), "Sai kích thước đặc trưng llm_3d!"
assert S_fused_3d.shape == (3769, 768), "Sai kích thước đặc trưng fused_3d!"

# Từ điển nội dung chỉ chứa riêng llm_3d và fused_3d phục vụ E-RQ3
CONTENTS_3D = {
    "llm_3d": S_llm_3d_norm,
    "fused_3d": S_fused_3d
}
    
print(f"Tổng số khóa học: {len(course_ids)}, Tổng số học viên: {len(user_cold_ids)}")
for k, v in CONTENTS_3D.items():
    print(f"Đã tải thành công đầu vào '{k}': kích thước {v.shape}")

MD5 llm_profile_no_prereq_followup_embeddings.npy: 1af436095463ad0af3ef1203ec0ebd6c
Tổng số khóa học: 3769, Tổng số học viên: 98220
Đã tải thành công đầu vào 'llm_3d': kích thước (3769, 384)
Đã tải thành công đầu vào 'fused_3d': kích thước (3769, 768)


In [5]:
# Chuẩn bị dữ liệu đánh giá tập test_cold
te_cold = pd.read_parquet('/kaggle/input/datasets/ngocccchannn/cold-scenario-data/test_cold.parquet')
tr_cold = pd.read_parquet('/kaggle/input/datasets/ngocccchannn/cold-scenario-data/train_cold.parquet')
va_cold = pd.read_parquet('/kaggle/input/datasets/ngocccchannn/cold-scenario-data/val_cold.parquet')

user2id = {str(u): i for i, u in enumerate(user_cold_ids)}
item2id = {str(c): i for i, c in enumerate(course_ids)}

def build_hist(df):
    hist_dict = {}
    for u, group in df.groupby('user_id'):
        u_str = str(u)
        if u_str in user2id:
            u_idx = user2id[u_str]
            item_idxs = [item2id[str(i)] for i in group['course_id'] if str(i) in item2id]
            if len(item_idxs) > 0:
                hist_dict[u_idx] = item_idxs
    return hist_dict

test_target_dict = build_hist(te_cold)
print(f'test_target_dict với {len(test_target_dict)} học viên.')

# Đặt tên riêng hist_tr (cho cold-val) và hist_trva (cho cold-test)
hist_tr = build_hist(tr_cold)
print(f'hist_tr (chỉ train) với {len(hist_tr)} học viên.')

hist_trva = build_hist(pd.concat([tr_cold, va_cold]))
print(f'hist_trva (train + val) với {len(hist_trva)} học viên.')

val_target_dict = build_hist(va_cold)
print(f"val_target_dict với {len(val_target_dict)} học viên.")

test_target_dict với 3340 học viên.
hist_tr (chỉ train) với 98220 học viên.
hist_trva (train + val) với 98220 học viên.
val_target_dict với 2092 học viên.


In [6]:
with open('/kaggle/input/datasets/ngocccchannn/cold-scenario/cold_items_val.json', 'r') as f:
    cold_val_ids = json.load(f)
with open('/kaggle/input/datasets/ngocccchannn/cold-scenario/cold_items_test.json', 'r') as f:
    cold_test_ids = json.load(f)

cold_val_ids = cold_val_ids.get("cold_items", cold_val_ids) if isinstance(cold_val_ids, dict) else cold_val_ids
cold_test_ids = cold_test_ids.get("cold_items", cold_test_ids) if isinstance(cold_test_ids, dict) else cold_test_ids

cand_cold_indices = [item2id[str(c)] for c in cold_test_ids if str(c) in item2id]
cand_bool_mask_a = np.zeros(3769, dtype=bool)
cand_bool_mask_a[cand_cold_indices] = True

cand_cold_indices_val = [item2id[str(c)] for c in cold_val_ids if str(c) in item2id]
cand_bool_mask_a_val = np.zeros(3769, dtype=bool)
cand_bool_mask_a_val[cand_cold_indices_val] = True

cand_bool_mask_b = np.ones(3769, dtype=bool)

cold_item_set = set(str(c) for c in cold_val_ids + cold_test_ids)
cold_idx = np.array(sorted(item2id[c] for c in cold_item_set if c in item2id))
# Kiểm tra assertion khóa lạnh bằng 0 - không có tín hiệu ID
assert np.allclose(g_i_cold[cold_idx], 0.0), "Lỗi: Khóa lạnh phải có embedding g_i = 0 đối với mô hình thuần ID!"

# Kiểm tra chặt chẽ tính tách biệt của kịch bản cold-item
tr_items = set(tr_cold.course_id.astype(str))
cv, ct = set(map(str, cold_val_ids)), set(map(str, cold_test_ids))
assert not (cv & tr_items) and not (ct & tr_items) and not (cv & ct), "Lỗi: Rò rỉ khóa học lạnh!"
assert set(va_cold.course_id.astype(str)) <= cv
assert set(te_cold.course_id.astype(str)) <= ct
assert set(va_cold.user_id) <= set(tr_cold.user_id)
assert set(te_cold.user_id) <= set(tr_cold.user_id)
print("Đã vượt qua toàn bộ kiểm tra assert không rò rỉ dữ liệu!")

Đã vượt qua toàn bộ kiểm tra assert không rò rỉ dữ liệu!


In [7]:
# Tách nhóm P-RQ3 (124 khóa trùng / 378 khóa mới) dựa trên dup_group_id
with open('/kaggle/input/datasets/ngocccchannn/embeddings-course-index/embeddings_course_index.json', 'r') as f:
    llm_meta = json.load(f)

i_tr = tr_cold["course_id"].astype(str).map(item2id).to_numpy()
warm_items_np = np.unique(i_tr)
_, dup_int = np.unique(llm_meta["dup_group_id"], return_inverse=True)
warm_dup_groups = set(dup_int[warm_items_np].tolist())

cold_test_dup_ids = [c for c in cold_test_ids if dup_int[item2id[str(c)]] in warm_dup_groups]
cold_test_new_ids = [c for c in cold_test_ids if dup_int[item2id[str(c)]] not in warm_dup_groups]
assert len(cold_test_dup_ids) == 124 and len(cold_test_new_ids) == 378, "Sai số lượng 124/378 khóa ứng viên P-RQ3!"

groups_prq3 = {
    "trung": torch.tensor([item2id[str(c)] for c in cold_test_dup_ids if str(c) in item2id]),
    "moi": torch.tensor([item2id[str(c)] for c in cold_test_new_ids if str(c) in item2id])
}
print(f"Xác nhận phân tách P-RQ3 thành công: {len(cold_test_dup_ids)} khóa trùng | {len(cold_test_new_ids)} khóa mới.")

Xác nhận phân tách P-RQ3 thành công: 124 khóa trùng | 378 khóa mới.


In [8]:
# Khởi tạo dữ liệu tensor cho DropoutNet
import torch.nn.functional as F

# Chuyển đổi các ma trận numpy sang torch.tensor và đưa lên thiết bị (GPU/CPU)
U_tensor = torch.tensor(g_u_cold, dtype=torch.float32, device=DEVICE)  # (98220, 64)
V_tensor = torch.tensor(g_i_cold, dtype=torch.float32, device=DEVICE)  # (3769, 64)

# Chỉ lấy 2.401 khóa học ấm có tương tác trong tr_cold để tính MSE
warm_np = np.unique(tr_cold.course_id.astype(str).map(item2id).to_numpy())   # 2.401 khóa
warm_cols = torch.tensor(warm_np, dtype=torch.long, device=DEVICE)
assert len(warm_cols) == 2401 and not (set(warm_cols.tolist()) & set(cold_idx.tolist())), "Lỗi tập warm_cols!"

# Điểm đích R_teacher chỉ tính 1 lần trên đúng 2.401 cột khóa học ấm có tương tác
R_teacher = U_tensor @ V_tensor[warm_cols].t()  # [98220, 2401]

print(f"Kích thước tensor U: {U_tensor.shape}")
print(f"Kích thước tensor V: {V_tensor.shape}")
print(f"Kích thước R_teacher (chỉ trên khóa ấm có tương tác): {R_teacher.shape}")

Kích thước tensor U: torch.Size([98220, 64])
Kích thước tensor V: torch.Size([3769, 64])
Kích thước R_teacher (chỉ trên khóa ấm có tương tác): torch.Size([98220, 2401])


In [9]:
# Định nghĩa lớp DropoutNetItemCold
class DropoutNetItemCold(nn.Module):
    def __init__(self, d=64, d_content=384, h=200):
        super().__init__()
        self.fU = nn.Sequential(nn.Linear(d, h), nn.Tanh(), nn.Linear(h, d))
        self.fV = nn.Sequential(nn.Linear(d + d_content, h), nn.Tanh(), nn.Linear(h, d))

    def forward(self, u_pref, v_pref, v_content):
        # u_pref: embedding học viên từ LightGCN nền
        # v_pref: embedding khóa học từ LightGCN nền (sẽ bị dropout về 0 khi mô phỏng khóa lạnh)
        # v_content: embedding văn bản của khóa học
        return self.fU(u_pref), self.fV(torch.cat([v_pref, v_content], -1))

# In số lượng tham số cho đầu vào 384 chiều và 768 chiều
for dim_c in [384, 768]:
    tmp_m = DropoutNetItemCold(d=64, d_content=dim_c, h=200)
    n_params = sum(p.numel() for p in tmp_m.parameters() if p.requires_grad)
    print(f"Số tham số DropoutNetItemCold (d_content={dim_c}): {n_params:,}")

Số tham số DropoutNetItemCold (d_content=384): 128,528
Số tham số DropoutNetItemCold (d_content=768): 205,328


In [10]:
# Suy luận giữ nguyên V_tensor (khóa lạnh đã có sẵn g_i = 0)
@torch.no_grad()
def dn_vectors(model, content_t):
    model.eval()
    return model(U_tensor, V_tensor, content_t)

def dn_ndcg_val(model, content_t, seed):
    u_e, v_e = dn_vectors(model, content_t)
    r = evaluate(lambda ub: u_e[ub.to(DEVICE)] @ v_e.t(), val_target_dict, mask=hist_tr,
                 cand=cand_bool_mask_a_val, ks=(20,), tie_seed=seed, bs=1024)
    return float(r["NDCG@20"].mean())

def train_dn(content_t, cfg, seed, max_steps=1000, eval_every=10, patience=10):
    set_seed(seed)
    model = DropoutNetItemCold(d=64, d_content=content_t.shape[1], h=200).to(DEVICE)
    opt = optim.Adam(model.parameters(), lr=cfg["lr"], weight_decay=cfg["wd"])
    best, best_state, bad, best_step = -1.0, None, 0, 0

    for step in range(1, max_steps + 1):
        model.train()
        opt.zero_grad()
        # Chỉ dropout và tính MSE trên 2.401 cột khóa học ấm có tương tác (warm_cols)
        keep = (torch.rand(len(warm_cols), 1, device=DEVICE) > cfg["p"]).float()
        u_p, v_p = model(U_tensor, V_tensor[warm_cols] * keep, content_t[warm_cols])
        loss = F.mse_loss(u_p @ v_p.t(), R_teacher)
        loss.backward()
        opt.step()

        if step % eval_every == 0:
            v = dn_ndcg_val(model, content_t, seed)
            if v > best:
                best, best_state, bad, best_step = v, copy.deepcopy(model.state_dict()), 0, step
            else:
                bad += 1
                if bad >= patience:
                    break

    model.load_state_dict(best_state)
    return model, best, best_step

In [11]:
# Đã khởi tạo riêng hist_tr và hist_trva ở Cell 4 để tránh ghi đè mặt nạ lịch sử.
print(f"Kiểm tra mặt nạ: hist_tr có {len(hist_tr)} học viên, hist_trva có {len(hist_trva)} học viên.")

Kiểm tra mặt nạ: hist_tr có 98220 học viên, hist_trva có 98220 học viên.


In [ ]:
@torch.no_grad()
def eval_subgroups(score_fn, targets, mask, cand, groups, ks=(5, 10, 20), tie_seed=0, bs=1024):
    """
    X xếp hạng trên TOÀN BỘ tập ứng viên cand (502 khóa cold-test). Với mỗi nhóm, chỉ số chỉ
    tính trên các đích thuộc nhóm, và chỉ cho học viên có ít nhất một đích trong nhóm.
    """
    users = sorted(targets); K = max(ks)
    g = torch.Generator().manual_seed(tie_seed)
    disc = 1.0 / np.log2(np.arange(2, K + 2))
    cand_t = torch.as_tensor(cand) if cand is not None else None
    
    out = {n: {f"{m}@{k}": [] for m in ("Recall", "NDCG", "HR") for k in ks} for n in groups}
    
    for s in range(0, len(users), bs):
        ub = users[s:s + bs]
        sc = score_fn(torch.tensor(ub)).float().cpu().clone()
        if cand_t is not None:
            sc[:, ~cand_t] = -float("inf")
        for r, u in enumerate(ub):
            if mask.get(u):
                sc[r, list(mask[u])] = -float("inf")
        
        top = topk_random_ties(sc, K, g)
        
        for r, u in enumerate(ub):
            for n, gset in groups.items():
                gt = [i for i in targets[u] if i in gset]
                if not gt:
                    continue
                hit = np.isin(top[r], gt).astype(float)
                for k in ks:
                    h = hit[:k]
                    idcg = disc[:min(len(gt), k)].sum()
                    out[n][f"Recall@{k}"].append(h.sum() / len(gt))
                    out[n][f"NDCG@{k}"].append((h * disc[:k]).sum() / idcg)
                    out[n][f"HR@{k}"].append(float(h.sum() > 0))
                    
    return {n: {m: np.array(v) for m, v in d.items()} for n, d in out.items()}

In [13]:
os.makedirs("scores", exist_ok=True)

# Danh sách 5 seed chuẩn 
SEEDS = [42, 2023, 2024, 1007, 999]

def run_5_seeds(name, cfg, tag=""):
    if 'CONTENTS' in globals() and name in CONTENTS:
        c = torch.tensor(CONTENTS[name], dtype=torch.float32, device=DEVICE)
    else:
        c = torch.tensor(CONTENTS_3D[name], dtype=torch.float32, device=DEVICE)
    
    full, grp, nd, rc = [], {"trung": [], "moi": []}, [], []
    
    for s in SEEDS:
        # Huấn luyện với cấu hình cố định và seed s
        model, _, _ = train_dn(c, cfg, seed=s)
        u_e, v_e = dn_vectors(model, c)
        
        # Hàm tính điểm score cho evaluation
        fn = lambda ub: u_e[ub.to(DEVICE)] @ v_e.t()
        
        # Đánh giá trên tập (a) - Khóa lạnh chính
        r = evaluate(fn, test_target_dict, cand=cand_bool_mask_a, mask=hist_trva,
                     ks=(5, 10, 20), tie_seed=s, bs=1024)
                     
        # Đánh giá phân nhóm P-RQ3 (trung / moi)
        sub = eval_subgroups(fn, test_target_dict, hist_trva, cand_bool_mask_a,
                             groups_prq3, ks=(20,), tie_seed=s)
                             
        full.append(r["NDCG@20"])
        nd.append(r["NDCG@20"].mean())
        rc.append(r["Recall@20"].mean())
        
        for g in grp:
            grp[g].append(sub[g]["NDCG@20"])
            
    # Tạo khóa tên tệp chuẩn khớp tuyệt đối với quy ước của final_stats.ipynb
    key = f"dropoutnet_{name}{tag}"
    
    # Lưu tệp điểm NDCG@20 toàn tập trung bình 5 seed vào thư mục scores/
    np.save(f"scores/user_ndcg20_{key}.npy", np.mean(full, axis=0))
    
    # Lưu tệp điểm phân nhóm P-RQ3 trung bình 5 seed vào scores/
    for g in grp:
        np.save(f"scores/user_ndcg20_{key}_{g}.npy", np.mean(grp[g], axis=0))
        
    print(f"[{key}]: NDCG@20 = {np.mean(nd):.4f} ± {np.std(nd, ddof=1):.4f}, "
          f"R@20 = {np.mean(rc):.4f} ± {np.std(rc, ddof=1):.4f} | Config: {cfg}")
          
    return float(np.mean(nd))

In [ ]:
CFG_5D = {
    "llm_3d": {"lr": 0.005, "p": 0.1, "wd": 1e-4},
    "fused_3d": {"lr": 0.0005, "p": 0.1, "wd": 1e-3}
}

print("Đã nạp cấu hình chuẩn cho llm_3d và fused_3d:")
for k, v in CFG_5D.items():
    print(f"  - {k}: {v}")
    
# Chạy đánh giá cho fused_3d và llm_3d
run_5_seeds("fused_3d", CFG_5D["fused_3d"])
m_llm3d = run_5_seeds("llm_3d", CFG_5D["llm_3d"])

# Kiểm tra tái lập kết quả llm_3d có khớp kỳ vọng ~0.0751 không
print("Kiểm tra llm_3d tái lập 0.0751:", abs(m_llm3d - 0.0751) < 5e-4)

Đã nạp cấu hình chuẩn cho llm_3d và fused_3d:
  - llm_3d: {'lr': 0.005, 'p': 0.1, 'wd': 0.0001}
  - fused_3d: {'lr': 0.0005, 'p': 0.1, 'wd': 0.001}
[dropoutnet_fused_3d]: NDCG@20 = 0.0886 ± 0.0081, R@20 = 0.2084 ± 0.0229 | Config: {'lr': 0.0005, 'p': 0.1, 'wd': 0.001}
[dropoutnet_llm_3d]: NDCG@20 = 0.0751 ± 0.0072, R@20 = 0.1822 ± 0.0156 | Config: {'lr': 0.005, 'p': 0.1, 'wd': 0.0001}
Kiểm tra llm_3d tái lập 0.0751: True


In [15]:
# Đường dẫn tới một file điểm cần kiểm tra
file_path = "/kaggle/working/scores/user_ndcg20_dropoutnet_llm_3d_trung.npy"

if os.path.exists(file_path):
    data = np.load(file_path)
    print(f"--- THÔNG TIN TỆP: {file_path} ---")
    print(f"Kích thước (Shape): {data.shape}")  # Kỳ vọng mong đợi: (1028,) đối với nhóm trùng
    print(f"Kiểu dữ liệu (Dtype): {data.dtype}")
    print(f"Giá trị NDCG@20 trung bình của nhóm: {np.mean(data):.4f}")
    print(f"5 giá trị điểm NDCG@20 đầu tiên của học viên: {data[:5]}")
else:
    print(f"Không tìm thấy tệp tại đường dẫn: {file_path}. Vui lòng kiểm tra lại quá trình chạy cell lưu file.")

--- THÔNG TIN TỆP: /kaggle/working/scores/user_ndcg20_dropoutnet_llm_3d_trung.npy ---
Kích thước (Shape): (1028,)
Kiểu dữ liệu (Dtype): float64
Giá trị NDCG@20 trung bình của nhóm: 0.0613
5 giá trị điểm NDCG@20 đầu tiên của học viên: [0.33876421 0.09553405 0.         0.14309367 0.        ]


In [ ]:
# Kiểm tra độ bền: cấu hình tốt nhất theo cold-val trong log 20 cấu hình 
for d in ("scores", "results_dropoutnet"):
    os.makedirs(d, exist_ok=True)

LLM_NPY = ("/kaggle/input/datasets/ngocccchannn/llm-embeddings/llm_profile_embeddings.npy")
S_llm_norm = nz(np.load(LLM_NPY).astype(np.float32))
CONTENTS = {"raw": S_raw_norm, 
            "llm": S_llm_norm,
            "fused": np.concatenate([S_raw_norm, S_llm_norm], axis=1), **CONTENTS_3D}

cf = lambda lr, p, wd: dict(lr=lr, p=p, wd=wd)
TOP3 = {"raw": [cf(0.005, 0.8, 1e-4), cf(0.01, 0.8, 1e-5), cf(0.005, 0.3, 1e-4)],
        "llm": [cf(0.005, 0.1, 1e-4), cf(0.001, 0.3, 1e-5), cf(0.005, 0.3, 1e-4)],
        "fused": [cf(0.0005, 0.1, 1e-3), cf(0.0005, 0.1, 1e-4), cf(0.005, 0.1, 1e-4)]}
log = {}

for name, cands in TOP3.items():
    if name not in CONTENTS:
        print(f"Bỏ qua '{name}' vì không có trong CONTENTS hiện tại.")
        continue
    c = torch.tensor(CONTENTS[name], dtype=torch.float32, device=DEVICE)

    # Chạy cold-val 3 seed cho từng ứng viên trong top 3 cấu hình
    val3 = [float(np.mean([train_dn(c, cfg, seed=s)[1] for s in [42, 2023, 2024]]))
            for cfg in cands]
    
    # Chọn cấu hình có giá trị trung bình cao nhất
    best = cands[int(np.argmax(val3))]
    log[name] = dict(val3=val3, chon=best)
    print(f"[{name}] cold-val 3 seed: {np.round(val3, 4)} -> chọn {best}")
    
    run_5_seeds(name, best, tag="_robust3s")

json.dump(log, open("results_dropoutnet/robust3s_log.json", "w"), indent=1)
print("==> Hoàn tất phân tích độ bền cho DropoutNet và đã lưu tệp _robust3s vào scores/!")

[raw] cold-val 3 seed: [0.0843 0.0874 0.0854] -> chọn {'lr': 0.01, 'p': 0.8, 'wd': 1e-05}
[dropoutnet_raw_robust3s]: NDCG@20 = 0.0835 ± 0.0109, R@20 = 0.1953 ± 0.0278 | Config: {'lr': 0.01, 'p': 0.8, 'wd': 1e-05}
[llm] cold-val 3 seed: [0.0999 0.0936 0.0973] -> chọn {'lr': 0.005, 'p': 0.1, 'wd': 0.0001}
[dropoutnet_llm_robust3s]: NDCG@20 = 0.0737 ± 0.0077, R@20 = 0.1859 ± 0.0147 | Config: {'lr': 0.005, 'p': 0.1, 'wd': 0.0001}
[fused] cold-val 3 seed: [0.0925 0.0911 0.0952] -> chọn {'lr': 0.005, 'p': 0.1, 'wd': 0.0001}
[dropoutnet_fused_robust3s]: NDCG@20 = 0.0853 ± 0.0085, R@20 = 0.2099 ± 0.0157 | Config: {'lr': 0.005, 'p': 0.1, 'wd': 0.0001}
==> Hoàn tất phân tích độ bền cho DropoutNet và đã lưu tệp _robust3s vào scores/!
